# VisXAI: Interactive Dashboard Demo

This notebook demonstrates `ExplanationDashboard`
(`visxai/visualizers/interactive.py`) &mdash; an `ipywidgets` dashboard that
compares two **pre-computed** explanations side by side on a shared,
adjustable color scale, with each panel **hover-linked** to an
XSMILES-style SMILES strip (`visxai/visualizers/hover_widget.py`).

**You need a live Jupyter kernel to see the widgets**, plus the `[viz]`
extra:

```
pip install -e ".[viz]"      # ipywidgets >= 8, anywidget
```

Unlike the other four notebooks in `examples/`, this one's primary output
is an interactive control surface, not a static SVG &mdash; a rendered copy
on GitHub or nbviewer will show the setup and the sanity-check output, but
the dashboard itself will appear blank.

Everything here runs **fully offline** &mdash; no network, no downloaded
checkpoint, no training loop beyond a 10-tree RandomForest on five
molecules.

## Three things worth knowing up front

1. **The dashboard re-renders; it never re-explains.** Every explanation
   is computed once, ahead of time, and registered. Moving a slider or
   switching a dropdown recolors and redraws those existing scores &mdash;
   it does not re-run an explainer. Live re-attribution would mean e.g.
   Integrated Gradients' `n_steps` forward/backward passes on every
   slider drag.
2. **It is paradigm-agnostic.** This demo uses the tree/fingerprint path
   because it's the fastest to set up offline, but the dashboard only ever
   touches `MoleculeRepresentation` and `Explanation` &mdash; the same two
   types every path emits. A sequence- or graph-path explanation drops in
   with no dashboard-side changes at all. The SMILES strip likewise works
   for all three, since `compute_atom_char_spans` derives its mapping from
   SMILES syntax rather than from any model.
3. **If the widgets don't appear at all**, check your `ipywidgets` version
   before debugging anything else. Widget rendering needs the frontend
   package to match the protocol version the kernel emits, and ipywidgets 7
   renders a *blank output area* on JupyterLab 4 / Notebook 7 &mdash; no
   error, nothing. That's why the `[viz]` extra pins `>= 8.0`.

In [1]:
from sklearn.ensemble import RandomForestClassifier

from visxai.explainers.tree_shap import TreeSHAPExplainer
from visxai.features.fingerprints import generate_morgan_representation
from visxai.models.sklearn_wrapper import SklearnModelWrapper
from visxai.visualizers.interactive import ExplanationDashboard

## The model

The same tiny 5-molecule synthetic dataset as `tree_demo.ipynb`. The labels are arbitrary (this is a toy set,
not a real structure-activity dataset) but give the classifier *some*
structure to key off, so the SHAP explanations aren't degenerate.

In [2]:
_SMILES = [
    "CCO",
    "c1ccccc1",
    "CC(=O)O",
    "CN1CCCC1",
    "CC(=O)Oc1ccccc1C(=O)O",
]
_LABELS = [0, 1, 0, 1, 1]
_N_BITS = 2048

fingerprint_matrix = [
    generate_morgan_representation(smi, n_bits=_N_BITS).fingerprint_array
    for smi in _SMILES
]

clf = RandomForestClassifier(n_estimators=10, random_state=42)
clf.fit(fingerprint_matrix, _LABELS)
wrapper = SklearnModelWrapper(clf)
print("Trained RandomForestClassifier on", len(_SMILES), "molecules.")

Trained RandomForestClassifier on 5 molecules.


[20:32:20] DEPRECATION WARNING: please use MorganGenerator
[20:32:20] DEPRECATION WARNING: please use MorganGenerator
[20:32:20] DEPRECATION WARNING: please use MorganGenerator
[20:32:20] DEPRECATION WARNING: please use MorganGenerator
[20:32:20] DEPRECATION WARNING: please use MorganGenerator


## Computing the explanations up front

Three molecules on one axis, and on the other axis TreeSHAP's two
`bond_score_mode` settings, which are genuinely different explanations of
the same prediction:

- **`"duplicate"`** (the default) gives each bond the bit's *full* SHAP
  score, independently of the atom distribution. `sum(atom_scores) +
  sum(bond_scores)` therefore double-counts each bit.
- **`"split"`** divides each bit's score across the *combined* set of its
  unique atoms and bonds, so the two sums together equal the bit's own
  SHAP value &mdash; at the cost of shrinking every `atom_scores` value
  relative to `"duplicate"`.

Where the two modes *do* diverge, that magnitude difference is exactly
what makes the shared color scale below worth having: without one, the
same atom renders a different color in each panel purely because the
*other* panel's scores differ.

Watch for one honest wrinkle in the sums printed below: **ibuprofen's two
modes come out identical.** That isn't a bug &mdash; all four of its
SHAP-contributing bits happen to be radius-0 environments (a single atom,
no bonds at all), and a bit with no bonds has nothing to split across, so
`"split"` degenerates to `"duplicate"` exactly. Aspirin (4 of its 9
contributing bits carry bonds) and caffeine (1 of 4) do differ. A useful
reminder that `bond_score_mode` only bites where the active bits actually
span bonds.

In [3]:
molecules = {
    "aspirin": "CC(=O)Oc1ccccc1C(=O)O",
    "ibuprofen": "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "caffeine": "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
}

explainers = {
    "TreeSHAP (duplicate)": TreeSHAPExplainer(bond_score_mode="duplicate"),
    "TreeSHAP (split)": TreeSHAPExplainer(bond_score_mode="split"),
}

# Featurize once per molecule, then explain it with each explainer. This is
# the loop the dashboard deliberately does NOT do for you -- everything it
# renders is already computed by the time it's registered.
results = {}
for name, smiles in molecules.items():
    mol_rep = generate_morgan_representation(smiles, n_bits=_N_BITS)
    results[name] = (
        mol_rep,
        {
            label: explainer.explain(wrapper, mol_rep)
            for label, explainer in explainers.items()
        },
    )

for name, (mol_rep, explanations) in results.items():
    print(f"{name} ({mol_rep.mol.GetNumAtoms()} atoms, {mol_rep.mol.GetNumBonds()} bonds)")
    for label, explanation in explanations.items():
        atom_sum = sum(explanation.atom_scores.values())
        bond_sum = sum(explanation.bond_scores.values())
        print(f"    {label:<22} atoms {atom_sum:+.6f}   bonds {bond_sum:+.6f}")

aspirin (13 atoms, 13 bonds)
    TreeSHAP (duplicate)   atoms +0.232778   bonds +0.142500
    TreeSHAP (split)       atoms +0.168553   bonds +0.064225
ibuprofen (15 atoms, 15 bonds)
    TreeSHAP (duplicate)   atoms +0.071111   bonds +0.000000
    TreeSHAP (split)       atoms +0.071111   bonds +0.000000
caffeine (14 atoms, 15 bonds)
    TreeSHAP (duplicate)   atoms +0.081111   bonds +0.010000
    TreeSHAP (split)       atoms +0.077778   bonds +0.003333


[20:32:20] DEPRECATION WARNING: please use MorganGenerator
[20:32:20] DEPRECATION WARNING: please use MorganGenerator
[20:32:20] DEPRECATION WARNING: please use MorganGenerator


## Building the dashboard

`from_results()` takes a nested
`{molecule: (mol_rep, {explainer: explanation})}` dict &mdash; which is
exactly the shape the loop above produced. Note that each `mol_rep` is
written **once per molecule**, not once per explanation: a representation
is a property of the molecule, not of any one explanation of it.

In [4]:
dashboard = ExplanationDashboard.from_results(results, width=380, height=300)

print("molecules: ", dashboard.molecules)
print("explainers:", dashboard.explainers)

molecules:  ['aspirin', 'ibuprofen', 'caffeine']
explainers: ['TreeSHAP (duplicate)', 'TreeSHAP (split)']


The equivalent using the lower-level `add()` primitive, which
`from_results()` just delegates to &mdash; useful when results arrive
incrementally rather than all at once:

```python
dashboard = ExplanationDashboard()
for name, (mol_rep, explanations) in results.items():
    for label, explanation in explanations.items():
        dashboard.add(name, mol_rep, explanation, explainer=label)
```

## The dashboard

Run the cell below in a live kernel to get the interactive view:

- **Two panels**, each with its own molecule and explainer dropdown. They
  open on the same molecule under *different* explainers, since comparing
  two explainers on one structure is the common case. Pass `n_panels=1`
  for a single-panel view instead.
- **Each panel is hover-linked** (`hover_link=True` by default): the
  structure sits above an **XSMILES-style SMILES strip** &mdash; the SMILES
  string laid out character by character, with a score bar above each one.
  - **Hover an atom in the structure** and the characters spelling it light
    up in the strip; every *other* column dims so the match is unmissable.
  - **Hover a character in the strip** and the corresponding atom outlines
    in the structure. The link works both ways.
  - A **readout follows the cursor**, showing the raw score *and* the
    normalized value that drives the color &mdash; they're different
    numbers, so seeing both is what lets you reconcile them with the legend.
  - **Most bonds have no SMILES character.** A plain single or aromatic
    bond is written by adjacency, not by a symbol, so only bonds with an
    explicit `=`, `#`, or ring digit get a strip column &mdash; 2 of
    aspirin's 13. The other 11 are still scored, still colored, and still
    hoverable; their readout says `no SMILES character` to explain why the
    strip stays dark. This is a hard limit of SMILES syntax, not a gap in
    the attribution.
  - Requires `anywidget` (`pip install "visxai[viz]"`). Without it the
    panel falls back to a static SVG **and warns** &mdash; it won't silently
    pretend to be linked.
- **Color scale, three modes:**
  - *Shared range &mdash; compare across panels* pools the range over the
    explanations **currently on screen**, recomputed whenever you change a
    selector. This is what makes the two panels comparable. It deliberately
    pools over the displayed panels rather than everything registered, so a
    molecule you never look at can't flatten the ones you do. The cost is
    that the panels become **coupled**: changing one panel's explainer can
    recolor the other, because they share one denominator.
  - *Per panel &mdash; best contrast in each* gives each panel the best
    contrast against its own scores, at the cost of colors no longer being
    comparable **between** panels. No coupling: one panel never affects the
    other.
  - *Manual range* hands control to the sliders, for clipping an outlier
    that is flattening everything else (the usual `vmin`/`vmax` need), or
    for pinning a range you want to reuse &mdash; drag until it looks right,
    then pass those numbers to `RDKitSVGVisualizer(atom_range=...)` for a
    static figure. The sliders only appear in this mode, and they start
    from whatever range the automatic mode was already using.

  **Which mode you start in follows the legend units, because the two are
  not independent.** A *normalized* legend prints the result of dividing by
  the range, never the range itself, so under a shared scale the colors can
  move while the tick labels stay at `-1.00`/`0`/`+1.00` &mdash; the picture
  changes with nothing on screen explaining why. Normalized therefore means
  "relative importance **within this molecule**", a per-panel idea, and that
  is what a default dashboard opens on. Choose raw units and the default
  becomes the shared range, where the ticks move with the pooling and stay
  readable.
- **Legend** switches what the color bar's min/max ticks print: the
  normalized `-1..+1` values that drive the color, or the raw scores.
- **Set the two panels to `aspirin` under different explainers** to see the
  numeric summaries diverge: `"duplicate"`'s sums double-count each bit,
  `"split"`'s don't.

In [5]:
dashboard.display()

## Sanity check (so this notebook is verifiable without a browser)

The widgets above can't be checked by an automated notebook run &mdash;
`nbclient` executes `display()` happily whether or not anything actually
renders. These assertions inspect the built widget tree directly instead,
so an automated notebook run catches a genuinely broken dashboard rather
than only a raised exception.

This reaches into `_panels`/`_controls`, which are private &mdash; fine for
a verification cell, but not the way to use the dashboard.

In [6]:
import re

# RDKitSVGVisualizer stamps a fresh random gradient id on every render, so
# two SVGs of identical data still differ as raw strings. Strip it before
# any did-this-change assertion -- otherwise "changed" is trivially true
# and these checks would pass even if nothing were wired up at all.
def strip_ids(svg):
    return re.sub(r"(visxaiLegendGradient\w*)-[0-9a-f]{8}", r"\1", svg)


# A panel's figure is a MoleculeHoverWidget when hover-linking is active and
# a plain ipywidgets.HTML otherwise, and the two expose their SVG
# differently. Reading through this keeps the checks valid either way.
def figure_svg(panel):
    figure = panel["figure"]
    return figure.payload.get("svg", "") if hasattr(figure, "payload") else figure.value


root = dashboard.widget()
panel_a, panel_b = dashboard._panels
print("hover-linking active:", dashboard._hover_link_active)

# Both panels rendered a real SVG and a summary.
for panel in (panel_a, panel_b):
    assert "<svg" in figure_svg(panel), "panel rendered no SVG"
    assert panel["summary"].value.startswith("<pre"), "panel rendered no summary"

# They opened on different explainers, so the view is a real comparison,
# and on a registered pair -- never an empty panel.
assert panel_a["explainer"].value != panel_b["explainer"].value
for panel in (panel_a, panel_b):
    assert dashboard.get(panel["molecule"].value, panel["explainer"].value) is not None

if dashboard._hover_link_active:
    # The SMILES strip: one entry per character, cross-linked to the
    # structure. Its colors must match the structure exactly -- both are
    # normalized together in Python precisely so they cannot drift.
    payload = panel_a["figure"].payload
    assert len(payload["chars"]) == len(payload["smiles"])
    checked = 0
    for entry in payload["chars"]:
        if entry["kind"] != "atom":
            continue
        found = re.search(
            r"<ellipse[^>]*class='atom-%d'[^>]*fill:(#[0-9A-F]{6})" % entry["index"],
            payload["svg"],
        )
        if found:
            checked += 1
            assert found.group(1) == entry["color"], "strip/structure color mismatch"
    assert checked > 0, "no atoms to cross-check"
    # The widget supplies its own instant readout, so the slow native
    # tooltip is deliberately off.
    assert "<title>" not in payload["svg"]
else:
    # Static fallback keeps the native <title> tooltips instead.
    assert "<title>" in figure_svg(panel_a)

# Changing a selector actually re-renders that panel.
before = panel_a["summary"].value
panel_a["molecule"].value = "caffeine"
assert panel_a["summary"].value != before, "selector change did not re-render"

# Under the default normalized legend the dashboard opens on PER-PANEL
# scaling, so one panel's colors never depend on what the other displays.
assert dashboard._controls["scale_mode"].value == "per-panel"
assert len(dashboard._displayed_explanations()) == 2

# Switching to the shared scale changes the rendering, and its pooled range
# covers the DISPLAYED panels only, so the third registered molecule cannot
# influence it.
before_both = [strip_ids(figure_svg(p)) for p in dashboard._panels]
dashboard._controls["scale_mode"].value = "shared"
assert any(
    b != strip_ids(figure_svg(p)) for b, p in zip(before_both, dashboard._panels)
), "shared scaling changed nothing"

# Shared scaling couples the panels: one pooled range means panel B's
# colors depend on what panel A is showing. Note it is *intermittent* --
# it only bites when the panel you touch owns one of the pooled extremes,
# so here, where ibuprofen supplies both ends, switching caffeine's
# explainer leaves the range untouched. Sometimes-recoloring is harder to
# reason about than always, which is part of why normalized units do not
# default to shared: a normalized legend prints the output of the division
# and never the divisor, so it cannot show the reader what moved.
#
# Per-panel scaling has no such coupling, ever -- that is what is asserted.
dashboard._controls["scale_mode"].value = "per-panel"
b_before = strip_ids(figure_svg(panel_b))
panel_a["explainer"].value = "TreeSHAP (split)"
assert strip_ids(figure_svg(panel_b)) == b_before, "per-panel must not couple panels"

# The legend-units switch relabels the color bar with raw scores.
ticks = lambda s: re.findall(r'font-family="sans-serif">([-+0-9.]+)</text>', s)
dashboard._controls["legend_units"].value = "normalized"
normalized_ticks = ticks(figure_svg(panel_a))
dashboard._controls["legend_units"].value = "raw"
assert ticks(figure_svg(panel_a)) != normalized_ticks, "legend units did not change"
dashboard._controls["legend_units"].value = "normalized"

# In manual mode the sliders drive both panels. Widen rather than narrow:
# narrowing clamps already-saturated scores to the same saturation, so it
# can legitimately be a no-op.
dashboard._controls["scale_mode"].value = "manual"
slider = dashboard._controls["atom_slider"]
assert slider.disabled is False, "manual mode left the slider disabled"
before_both = [strip_ids(figure_svg(p)) for p in dashboard._panels]
slider.value = (slider.min, slider.max)
assert all(
    b != strip_ids(figure_svg(p)) for b, p in zip(before_both, dashboard._panels)
), "slider change did not re-render both panels"

# The two panels' legend gradient ids must not collide -- both SVGs live in
# one document, and SVG id references resolve document-wide.
ids_a, ids_b = (
    set(re.findall(r'<linearGradient id="([^"]+)"', figure_svg(p)))
    for p in dashboard._panels
)
assert ids_a and ids_b and ids_a.isdisjoint(ids_b), "gradient ids collided"

# The single-panel layout builds and renders too.
single = ExplanationDashboard.from_results(results, n_panels=1)
single.widget()
assert len(single._panels) == 1, "n_panels=1 did not produce one panel"
assert "<svg" in figure_svg(single._panels[0])
assert [v for _, v in single._controls["scale_mode"].options] == ["shared", "manual"]

print("All dashboard sanity checks passed.")

hover-linking active: True
All dashboard sanity checks passed.


## What this doesn't do yet

- **No live re-explaining.** Registering a new molecule means computing its
  explanation yourself and calling `add()` again (which invalidates the
  cached widget, so the next `display()` picks it up).
- **No batch "run every explainer over every molecule" helper.** The loop
  a few cells up is written by hand on purpose &mdash; a helper that runs
  explainers isn't visualization code, and where it should live is its own
  open design question.
- **Every interaction is a full redraw.** RDKit's `MolDraw2DSVG` has no
  incremental API, so each change regenerates a complete new SVG string
  rather than patching the existing image.
- **No standalone web app.** A standalone served web app was deliberately dropped from the plan &mdash; which is also why `anywidget` was chosen over Panel. A
  small server wrapper could be added later if that changes.
- **Nothing tests appearance.** The Python suite covers the payload and
  widget state; `npm test` covers the hover/link logic in jsdom. jsdom has
  no layout engine, so colors, bar geometry, and readout placement are
  only ever checked as the values the code *sets* &mdash; never as pixels.
  If it looks wrong, no test will tell you.